In [13]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
from openai import OpenAI
import os 

client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url = "https://api.groq.com/openai/v1"
)


In [15]:
def llm(prompt):
    response = client.responses.create(
        model = "llama-3.1-8b-instant",
        input = prompt
    )
    return response.output_text

In [16]:
llm("Hey, What's up?")

"Not much. What's on your mind?"

In [17]:
question = "I just discovered this course. Can i join now?" 
answer = llm(question)
print(answer)

I'm happy to hear that you're excited about a course. However, I'm a large language model, I don't have specific information about a particular course you're referring to.

Could you please provide me with more details about the course you're interested in, such as:

1. What subject is the course about (e.g. programming, language learning, etc.)?
2. Who is offering the course (e.g. a school, online platform, etc.)?
3. Have you checked the course dates and duration?

This will help me better understand your query and provide a more accurate answer about joining the course.


In [18]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [19]:
prompt = f"""
Your task is to answer the questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [20]:
answer = llm(prompt)
print(answer)

You can join the course now, but if you want to receive a certificate, you need to submit your project while they're still accepting submissions.


In [21]:
import requests
import json

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
print(json.dumps(courses_raw, indent=2))

[
  {
    "course": "data-engineering-zoomcamp",
    "course_name": "Data Engineering Zoomcamp",
    "path": "/json/data-engineering-zoomcamp.json",
    "questions_count": 402
  },
  {
    "course": "stock-markets-analytics-zoomcamp",
    "course_name": "Stock Markets Analytics Zoomcamp",
    "path": "/json/stock-markets-analytics-zoomcamp.json",
    "questions_count": 93
  },
  {
    "course": "ai-dev-tools-zoomcamp",
    "course_name": "AI Dev Tools Zoomcamp",
    "path": "/json/ai-dev-tools-zoomcamp.json",
    "questions_count": 41
  },
  {
    "course": "llm-zoomcamp",
    "course_name": "LLM Zoomcamp",
    "path": "/json/llm-zoomcamp.json",
    "questions_count": 83
  },
  {
    "course": "mlops-zoomcamp",
    "course_name": "MLOps Zoomcamp",
    "path": "/json/mlops-zoomcamp.json",
    "questions_count": 255
  },
  {
    "course": "machine-learning-zoomcamp",
    "course_name": "ML Zoomcamp",
    "path": "/json/machine-learning-zoomcamp.json",
    "questions_count": 472
  }
]


In [22]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1346

In [23]:
documents[100]

{'id': '117164c439',
 'course': 'data-engineering-zoomcamp',
 'section': 'Module 1: Postgres, pgAdmin & Python ingestion',
 'question': 'pgAdmin: Create server dialog does not appear',
 'answer': 'pgAdmin has a new version. The create server dialog may not appear. Try using `Register` -> `Server` instead.'}

In [24]:
from minsearch import Index

index = Index(
    text_fields = ["question", "answer", "section"],
    keyword_fields = ["course"]
)

index.fit(documents)

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict = boost_dict,
        filter_dict = filter_dict,
        num_results = 5
    )

In [31]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [32]:
INSTRUCTIONS = """
Your task is to answer the questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with "I don't know."
"""

In [33]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""